In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pywt

In [2]:
#print db3 low pass filter
print(pywt.Wavelet('db3').dec_lo)

[0.03522629188570953, -0.08544127388202666, -0.13501102001025458, 0.45987750211849154, 0.8068915093110925, 0.33267055295008263]


In [3]:
db1 = np.array([1,1])/np.sqrt(2)
db2 = np.array([-0.12940952255126037, 0.2241438680420134, 0.8365163037378079, 0.48296291314453416])
db3 = np.array([0.03522629188570953, -0.08544127388202666, -0.13501102001025458, 0.45987750211849154, 0.8068915093110925, 0.33267055295008263])

In [4]:
def get_highpass_filter(low_pass_filter):
    '''Devuelve el filtro de paso alto asociado a un filtro de paso bajo.'''
    h = low_pass_filter.copy()
    h = np.flip(h)
    h[::2] *= -1
    return h

In [5]:
def extend_signal(signal, mode, n=None):
    '''Extiende la señal (dobla longitud) según el modo indicado.'''

    if n is None:
        n = len(signal)

    if mode == 'periodic':
        return np.concatenate((signal, signal[:n]))
    elif mode == 'symmetric':
        #TODO
        return np.concatenate((signal[1::-1], signal, signal[-2:-len(signal):-1]))
    elif mode == 'zero':
        return signal
    else:
        raise ValueError('Modo no soportado')

In [6]:
def apply_deconstruction(signal, low_pass_filter, mode='zero'):
    '''Aplica una descomposición wavelet a la señal.'''

    high_pass_filter = get_highpass_filter(low_pass_filter)
    
    _signal = extend_signal(signal, mode, len(low_pass_filter)-2)

    aprox = np.convolve(_signal, low_pass_filter)
    detail = np.convolve(_signal, high_pass_filter)

    #Downsampling
    aprox = aprox[1::2]
    detail = detail[1::2]

    return aprox, detail

    

In [7]:
def apply_construction(aprox, detail, low_pass):

    l = len(low_pass)

    high_pass_filter = get_highpass_filter(low_pass)

    _low_pass = np.flip(low_pass)
    _high_pass = np.flip(high_pass_filter)

    #upsampling
    _aprox = np.zeros(aprox.shape[0]*2)
    _detail = np.zeros(detail.shape[0]*2)

    _aprox[0::2] = aprox
    _detail[0::2] = detail

    r1 = np.convolve(_aprox, _low_pass)
    r2 = np.convolve(_detail, _high_pass)

    r = r1 + r2

    return r[l-2:-l+1]

In [8]:
signal = np.array([1,2,3,4,5,6,7,8,9,10,11,12])
filter = pywt.Wavelet('db5').dec_lo
aprox, detail = apply_deconstruction(signal, filter)
apply_construction(aprox, detail, filter)

array([ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12.])

In [9]:
signal = np.array([1,2,3,4,5,6,7,8,9,10,11,12])
low = db3
aprox, detail = apply_deconstruction(signal, low)
print('aprox', aprox/np.sqrt(2))
print('detail', detail)
reconstructed = apply_construction(aprox, detail, low)
print('reconstructed', reconstructed)

aprox [-0.0105986   0.05263477  1.81740117  3.81740117  5.81740117  7.81740117
 10.25408802  9.43427114]
detail [ 1.41550403e-01  3.52262919e-02  0.00000000e+00  0.00000000e+00
 -4.44089210e-16  8.88178420e-16 -5.83220188e+00  1.41278450e+00]
reconstructed [ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12.]


In [10]:
filter ='db2'
low = np.array(pywt.Wavelet(filter).dec_lo)
cA, cD = pywt.dwt(signal, filter, mode='periodic')
print('cA', cA/np.sqrt(2))
print('cD', cD)

apply_construction(cA, cD, low)

cA [10.83012702  1.6339746   3.6339746   5.6339746   7.6339746   9.6339746
 10.83012702]
cD [-4.24264069e+00  1.66533454e-16  3.33066907e-16  2.22044605e-16
  2.22044605e-16  4.44089210e-16 -4.24264069e+00]


array([ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12.])

In [11]:
low

array([-0.12940952,  0.22414387,  0.8365163 ,  0.48296291])

In [12]:
signal = np.array([1,2,3,4,5,6,7,8,9,10,11,12])
low_pass_filter = db2
a, d = apply_deconstruction(signal, low_pass_filter,'periodic')
print(a)
print(d)
apply_construction(a, d, low_pass_filter)

[-0.03467518  2.31078903  5.13921616  7.96764328 10.79607041 13.62449753
 15.31611251  2.15599552]
[-1.29409523e-01  2.22044605e-16  4.44089210e-16  4.44089210e-16
  0.00000000e+00  0.00000000e+00 -4.24264069e+00 -5.77697259e-01]


array([ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12.,  1.,
        2.])

In [13]:
signal = np.array([1,2,3,4,5,6,7,8])
cA, cD = pywt.dwt(signal, 'db1', mode='periodic')
print(cA/np.sqrt(2))
print(cD)

[1.5 3.5 5.5 7.5]
[-0.70710678 -0.70710678 -0.70710678 -0.70710678]


In [14]:
l = db2
s = np.array([1,2,3,4,5,6,7,8])
l

array([-0.12940952,  0.22414387,  0.8365163 ,  0.48296291])

In [15]:
np.convolve(l,s, mode='full')

array([-0.12940952, -0.03467518,  0.89657547,  2.31078903,  3.7250026 ,
        5.13921616,  6.55342972,  7.96764328, 10.54654255, 10.07287082,
        3.86370331])

In [16]:
np.convolve(l,s, mode='same')

array([-0.03467518,  0.89657547,  2.31078903,  3.7250026 ,  5.13921616,
        6.55342972,  7.96764328, 10.54654255])

In [17]:
np.convolve(l,s, mode='valid')

array([2.31078903, 3.7250026 , 5.13921616, 6.55342972, 7.96764328])

In [18]:
s = np.array([1,5,3,3,5,6,7,8])
cA, cD = pywt.dwt(s, 'db2', mode='periodic')
print(cA)
print(cD)

[9.64996708 4.94974747 4.30269986 7.96764328 9.64996708]
[-4.27731586e+00 -1.89468691e-01  2.24143868e-01  2.22044605e-16
 -4.27731586e+00]


In [19]:
f='db2'
signal = np.array([1,5,3,3,5,6,7,8])
filter = np.array(pywt.Wavelet(f).dec_lo)
a, d = apply_deconstruction(signal, filter, 'periodic')
print(a)
print(d)
print('------------------------------')
apply_construction(np.copy(a), np.copy(d), filter)

[-0.42290374  4.94974747  4.30269986  7.96764328  9.64996708  4.66554443]
[-1.57829826e+00 -1.89468691e-01  2.24143868e-01  4.44089210e-16
 -4.27731586e+00 -1.25012886e+00]
------------------------------


array([1., 5., 3., 3., 5., 6., 7., 8., 1., 5.])

In [20]:
l = filter
h = get_highpass_filter(l)
k = 1
s = np.array([1,5,3,3,5,6,7,8])
s = np.concatenate((s,s))

a=np.zeros(4)
d=np.zeros(4)
for i in range(4):
    a[i] = l[3]*s[0+2*i]+l[2]*s[1+2*i]+l[1]*s[2+2*i]+l[0]*s[3+2*i]
    d[i] = h[3]*s[0+2*i]+h[2]*s[1+2*i]+h[1]*s[2+2*i]+h[0]*s[3+2*i]

print(a,d)

[4.94974747 4.30269986 7.96764328 9.64996708] [-1.89468691e-01  2.24143868e-01  4.44089210e-16 -4.27731586e+00]


In [21]:
l[3]*a[0]+l[1]*a[3]+h[3]*d[0]+h[1]*d[3]

1.0000000000000004

In [22]:
l[3]*a[0]+l[1]*a[3]

4.553525403784439

In [23]:
l[2]*a[0]+l[0]*a[3] + h[2]*d[0]+h[0]*d[3]

5.000000000000001

In [24]:
l[1]*a[0]+l[3]*a[1] + h[1]*d[0]+h[3]*d[1]+h[0]*d[2]+h[2]*d[3]

3.958734122634726

In [25]:
_low_pass = np.flip(l)
_a = np.zeros(a.shape[0]*2)
_a[0::2] = a
_a

array([4.94974747, 0.        , 4.30269986, 0.        , 7.96764328,
       0.        , 9.64996708, 0.        ])

In [26]:
np.convolve(_a, _low_pass, mode='full')

array([ 2.39054446,  4.14054446,  3.1875    ,  2.95873412,  4.8125    ,
        6.10825318,  6.4464746 ,  7.04126588,  2.16298095, -1.24879763,
        0.        ])

In [27]:
signal = np.array([1,5,3,3,5,6,7,8,2,3,4,5,6,7,3,1,2,3,5,6,3,2,3,4])
a,c = apply_deconstruction(signal, db2, 'periodic')
print(a)
print(c)
apply_construction(a,c, db2)

[-0.42290374  4.94974747  4.30269986  7.96764328 10.13292999  3.7250026
  6.55342972  9.29641369  2.34546421  3.81973694  7.84752495  3.27671486
  4.37205021  4.66554443]
[-1.57829826e+00 -1.89468691e-01  2.24143868e-01  4.44089210e-16
 -2.47487373e+00  0.00000000e+00  4.44089210e-16 -3.18878214e-01
 -3.88228568e-01  3.53553391e-01 -4.48287736e-01 -2.58819045e-01
 -2.86310230e+00 -1.25012886e+00]


array([1., 5., 3., 3., 5., 6., 7., 8., 2., 3., 4., 5., 6., 7., 3., 1., 2.,
       3., 5., 6., 3., 2., 3., 4., 1., 5.])

In [28]:
cA, cD = pywt.dwt(signal, 'db6', mode='periodic')
db4 = np.array(pywt.Wavelet('db6').dec_lo)
apply_construction(cA,cD, db4)

array([1., 5., 3., 3., 5., 6., 7., 8., 2., 3., 4., 5., 6., 7., 3., 1., 2.,
       3., 5., 6., 3., 2., 3., 4.])

In [29]:
cA, cD

(array([ 1.62175087,  6.80068031,  5.37094176,  4.20848277,  4.01473307,
         4.72880244,  5.01881305, 10.81881712,  6.32170211,  4.29773192,
         8.95943922,  6.42746313,  1.62175087,  6.80068031,  5.37094176,
         4.20848277,  4.01473307]),
 array([-2.41818628, -1.71207112,  1.22111865, -0.41579882, -2.51275254,
         0.32368905, -0.30110466, -1.39682424,  1.32717242, -0.35579679,
        -1.06923233,  0.94582563, -2.41818628, -1.71207112,  1.22111865,
        -0.41579882, -2.51275254]))

In [30]:
a, d = apply_deconstruction(signal, db4, 'periodic')
a,d

(array([-6.09247916e-04, -1.77129590e-02,  1.59453823e-01, -6.29790368e-01,
         1.70161528e+00,  4.72880244e+00,  5.01881305e+00,  1.08188171e+01,
         6.32170211e+00,  4.29773192e+00,  8.95943922e+00,  6.42746313e+00,
         1.62175087e+00,  6.80068031e+00,  5.37094176e+00,  4.20848277e+00,
         4.01473307e+00,  4.71507991e+00,  5.12124956e+00,  1.04635572e+01,
         7.18579520e+00,  1.70695316e+00]),
 array([-0.06307983, -2.29116975,  1.49778093, -0.53100209, -2.49041161,
         0.32368905, -0.30110466, -1.39682424,  1.32717242, -0.35579679,
        -1.06923233,  0.94582563, -2.41818628, -1.71207112,  1.22111865,
        -0.41579882, -2.51275254, -1.09710279,  0.00660533, -0.01497593,
         0.05009463, -0.01648637]))

In [31]:
ff = 'db4'
filter = np.array(pywt.Wavelet(ff).dec_lo)
a,d=apply_deconstruction(signal, filter, 'periodic')
print(a, d)
cA, cD = pywt.dwt(signal, ff, mode='periodic')
cA, cD

[-0.020104    0.03402893  0.12321228  5.00857776  4.79363557  9.43026327
  8.03841141  4.170632    7.58885309  8.07013658  1.9111187   5.14863432
  6.81980035  3.93556273  3.673732    4.86317589  5.82217437  5.44096849] [-0.4370425  -1.72898134  1.1813878  -0.28568385 -3.35839829  1.22077046
 -0.30436289 -1.73121035  1.31121493  0.2291496  -1.68330739  1.03388756
 -2.82820819 -0.98587617  1.01806355 -3.44658733  1.14597454 -0.25028508]


(array([6.81980035, 3.93556273, 3.673732  , 5.00857776, 4.79363557,
        9.43026327, 8.03841141, 4.170632  , 7.58885309, 8.07013658,
        1.9111187 , 5.14863432, 6.81980035, 3.93556273, 3.673732  ]),
 array([-2.82820819, -0.98587617,  1.01806355, -0.28568385, -3.35839829,
         1.22077046, -0.30436289, -1.73121035,  1.31121493,  0.2291496 ,
        -1.68330739,  1.03388756, -2.82820819, -0.98587617,  1.01806355]))

In [32]:
apply_construction(a,d, filter)

array([1., 5., 3., 3., 5., 6., 7., 8., 2., 3., 4., 5., 6., 7., 3., 1., 2.,
       3., 5., 6., 3., 2., 3., 4., 1., 5., 3., 3., 5., 6.])

$d_k = \sqrt{2} \sum_{n \in \mathbf{Z}} \beta_{n}c_{2k-n}$<br><br>
$\beta_{n} = (-1)^{n+1}\alpha_{1-n}$

In [33]:
# beta empieza en idice 1 y va hasta el -L+1
np.log2(8)

3.0

In [34]:
def extend(signal):
    exponent = np.ceil(np.log2(len(signal)))
    n = 2**exponent-len(signal)
    return np.concatenate((signal, signal[:n]))

def apply_deconstruction(signal, low_pass_filter, mode='periodic'):
    '''Aplica una descomposición wavelet a la señal.'''

    high_pass_filter = get_highpass_filter(low_pass_filter)
    
    _signal = extend_signal(signal, mode, len(low_pass_filter)-2)

    aprox = np.convolve(_signal, low_pass_filter, mode='valid')
    detail = np.convolve(_signal, high_pass_filter, mode='valid')

    aprox = aprox[::2]
    detail = detail[::2]

    return aprox, detail

    

In [35]:
for i in range(5,10):
    s1 = np.ones(i)
    s2 = np.array([1,2,3,4])
    print(len(np.convolve(s1,s2)), len(s1)+len(s2)-1)   

8 8
9 9
10 10
11 11
12 12


In [36]:
for i in range(1,5):
    print('------{}------'.format(i))
    print((i*2+11-1)/2)
    f = 'db'+str(i)
    s2 = np.array([1,2,3,4,5,6, 7, 8, 9, 10,11])
    cA, cD = pywt.dwt(s2, f, mode='periodic')
    print("periodic", f, len(cA), len(cD))
    cA, cD = pywt.dwt(s2, f, mode='symmetric')
    print("symmetric", f, len(cA), len(cD))
    cA, cD = pywt.dwt(s2, f, mode='zero')
    print("zero", f, len(cA), len(cD))
  

------1------
6.0
periodic db1 6 6
symmetric db1 6 6
zero db1 6 6
------2------
7.0
periodic db2 7 7
symmetric db2 7 7
zero db2 7 7
------3------
8.0
periodic db3 8 8
symmetric db3 8 8
zero db3 8 8
------4------
9.0
periodic db4 9 9
symmetric db4 9 9
zero db4 9 9


In [37]:
for i in range(1,5):
    f = 'db'+str(i)
    cA, cD = pywt.dwt(s2, f, mode='zero')
    print(cA)
    ss = np.concatenate((s2,s2))
    n = np.ceil(np.log2(len(s2)))
    print(n)
    ss = ss[:int(2**n)]
    cc = np.convolve(ss, np.array(pywt.Wavelet(f).dec_lo), mode='full')
    print(cc)
    print("--------------------------")

[ 2.12132034  4.94974747  7.77817459 10.60660172 13.43502884  7.77817459]
4.0
[ 0.70710678  2.12132034  3.53553391  4.94974747  6.36396103  7.77817459
  9.19238816 10.60660172 12.02081528 13.43502884 14.8492424   8.48528137
  2.12132034  3.53553391  4.94974747  6.36396103  3.53553391]
--------------------------
[-0.03467518  2.31078903  5.13921616  7.96764328 10.79607041 15.1774118
  5.31259204]
4.0
[-0.12940952 -0.03467518  0.89657547  2.31078903  3.7250026   5.13921616
  6.55342972  7.96764328  9.38185685 10.79607041 12.21028397 15.04800228
 13.9966333   6.20916752  2.31078903  3.7250026   5.91567329  6.11443317
  2.41481457]
--------------------------
[-1.49886901e-02  7.44368080e-02  2.57019338e+00  5.39862050e+00
  8.22704763e+00  1.06327593e+01  1.61216026e+01  3.65937608e+00]
4.0
[ 3.52262919e-02 -1.49886901e-02 -2.00214692e-01  7.44368080e-02
  1.15597982e+00  2.57019338e+00  3.98440694e+00  5.39862050e+00
  6.81283407e+00  8.22704763e+00  9.64126119e+00  1.06679855e+01
  1.302